# 01. Limpieza y Normalizacion de Datos

Cargamos los datos extraidos de Instagram, TikTok y Trustpilot, los limpiamos y normalizamos para crear un dataset unificado listo para analisis.

**Fuentes de datos:**
- Instagram: publicaciones + comentarios (US + Europa)
- TikTok: videos + comentarios (US + Europa)
- Trustpilot: reviews (Lululemon + ALO Yoga)

In [1]:
import sys
sys.path.insert(0, "../../src")

import pandas as pd
import numpy as np
from cleaning import clean_text, safe_detect_lang, normalize_dates, add_engagement_rate
from plotting import setup_style
setup_style()

DATA_PROC = "../../datos/processed"
DATA_CLEAN = "../../datos/clean"

import os
os.makedirs(DATA_CLEAN, exist_ok=True)

## 1. Trustpilot Reviews

In [2]:
tp_lulu = pd.read_csv(f"{DATA_PROC}/trustpilot/trustpilot_lululemon.csv")
tp_alo = pd.read_csv(f"{DATA_PROC}/trustpilot/trustpilot_alo_yoga.csv")

tp_lulu["brand"] = "Lululemon"
tp_alo["brand"] = "ALO Yoga"

tp = pd.concat([tp_lulu, tp_alo], ignore_index=True)

# Limpieza
tp["text_clean"] = tp["text"].apply(clean_text)
tp["title_clean"] = tp["title"].apply(clean_text)
tp["published_date"] = pd.to_datetime(tp["published_date"], errors="coerce", utc=True).dt.tz_localize(None)
tp["experience_date"] = pd.to_datetime(tp["experience_date"], errors="coerce", utc=True).dt.tz_localize(None)
tp["year"] = tp["published_date"].dt.year
tp["month"] = tp["published_date"].dt.to_period("M").astype(str)

# Detectar idioma si no esta
if "language" not in tp.columns or tp["language"].isna().sum() > 0:
    tp["language"] = tp["text_clean"].apply(safe_detect_lang)

# Normalizar pais
tp["country"] = tp["reviewer_country"].str.upper().str.strip()

print(f"Trustpilot total: {len(tp)} reviews")
print(f"  Lululemon: {len(tp[tp['brand']=='Lululemon'])}")
print(f"  ALO Yoga:  {len(tp[tp['brand']=='ALO Yoga'])}")
print(f"\nNulos restantes:")
print(tp.isnull().sum()[tp.isnull().sum() > 0])
tp.head()

Trustpilot total: 383 reviews
  Lululemon: 188
  ALO Yoga:  195

Nulos restantes:
experience_date    2
dtype: int64


,review_id,title,text,rating,language,reviewer_name,reviewer_country,reviewer_num_reviews,is_verified,source,published_date,experience_date,brand,text_clean,title_clean,year,month,country
0,698f5a52b1c96308944c4701,El 8 de enero hice un pedido por valor…,El 8 de enero hice un pedido por valor de 500€...,1,es,CAROLINA,ES,4,False,Organic,2026-02-13 19:07:30,2026-01-08,Lululemon,El 8 de enero hice un pedido por valor de 500€...,El 8 de enero hice un pedido por valor…,2026,2026-02,ES
1,695cedaa651313c9714ed6e7,Hice un pedido el 25 de diciembre,"Hice un pedido el 25 de diciembre, me dijeron ...",1,es,Johanna,ES,2,False,Organic,2026-01-06 13:10:34,2025-12-25,Lululemon,"Hice un pedido el 25 de diciembre, me dijeron ...",Hice un pedido el 25 de diciembre,2026,2026-01,ES
2,695455e30f3d08e92102e95d,BUENAS TARDES,"BUENAS TARDES, \nNO TENGO UNA OPINION, \nSI TE...",5,es,fabulous Sundays,ES,1,False,Organic,2025-12-31 00:44:51,2025-12-30,Lululemon,"BUENAS TARDES, NO TENGO UNA OPINION, SI TENGO ...",BUENAS TARDES,2025,2025-12,ES
3,6939419888249807ec8ea87c,Estafa,He sufrido una estafa donde han utilizado mi t...,1,es,Patricia AC,ES,3,False,Organic,2025-12-10 11:47:04,2025-11-12,Lululemon,He sufrido una estafa donde han utilizado mi t...,Estafa,2025,2025-12,ES
4,679a2ac510e80efc85a9f337,Desastroso como no he experimentado en mi vida…,Desastroso como no he experimentado en mi vida...,1,es,rarafriend2,ES,18,False,Organic,2025-01-29 15:19:01,2025-01-09,Lululemon,Desastroso como no he experimentado en mi vida...,Desastroso como no he experimentado en mi vida…,2025,2025-01,ES


## 2. Instagram - Publicaciones

In [3]:
ig = pd.read_csv(f"{DATA_PROC}/instagram/ig_lululemon_publicaciones.csv")

# Limpieza
ig["caption_clean"] = ig["caption_std"].apply(clean_text)
ig["timestamp"] = pd.to_datetime(ig["timestamp_std"], errors="coerce")
ig["year"] = ig["timestamp"].dt.year
ig["month"] = ig["timestamp"].dt.to_period("M").astype(str)

# Renombrar columnas a esquema limpio
ig = ig.rename(columns={
    "likes_std": "likes",
    "comments_std": "comments",
    "url_std": "url",
})

# Engagement rate (sin views para posts, score ponderado)
ig = add_engagement_rate(ig, "likes", "comments")

# Eliminar duplicados por URL
n_before = len(ig)
ig = ig.drop_duplicates(subset="url", keep="first")
print(f"Instagram publicaciones: {n_before} -> {len(ig)} (eliminados {n_before - len(ig)} duplicados)")
print(f"Nulos restantes:")
print(ig.isnull().sum()[ig.isnull().sum() > 0])
ig.head()

Instagram publicaciones: 585 -> 577 (eliminados 8 duplicados)
Nulos restantes:
shortCode           1
timestamp_std       1
year                1
month               1
caption_std         1
hashtags            1
inputUrl            1
ownerUsername       1
ownerFullName      35
ownerId             1
locationName      385
locationId        385
type                1
productType         1
videoUrl          292
videoPlayCount    293
igPlayCount       293
musicInfo          32
timestamp           1
dtype: int64


,kind,url,shortCode,likes,comments,score,timestamp_std,year,month,caption_std,...,videoUrl,videoPlayCount,igPlayCount,musicInfo,rank_global,rank_type,label,caption_clean,timestamp,engagement_rate
0,reel,https://www.instagram.com/p/CiSz--iDg2N/,CiSz--iDg2N,131818,1603,136627,2022-09-09 17:13:56,2022.0,2022-09,My favorite workout of the week💪🏽\n•\n•\n#stro...,...,https://scontent-vie1-1.cdninstagram.com/o1/v/...,2520413.0,2520413.0,"{'artist_name': 'K CAMP', 'song_name': 'Bullse...",1,1,Reel 1,My favorite workout of the week💪🏽 • • #strongw...,2022-09-09 17:13:56,136627
1,reel,https://www.instagram.com/p/DShEoxLAGJW/,DShEoxLAGJW,61435,123,61804,2025-12-21 07:20:39,2025.0,2025-12,The sound of a Lululemon outfit ✨😌 @lululemon ...,...,https://scontent-vie1-1.cdninstagram.com/o1/v/...,3351430.0,3351430.0,"{'artist_name': 'melinatesibeauty', 'song_name...",2,2,Reel 2,The sound of a Lululemon outfit ✨😌 @lululemon ...,2025-12-21 07:20:39,61804
2,reel,https://www.instagram.com/p/DIE6ruFydgH/,DIE6ruFydgH,32705,119,33062,2025-04-05 19:40:25,2025.0,2025-04,This color!!🍵😍 Comment “SHOP” & I’ll send you ...,...,https://scontent-vie1-1.cdninstagram.com/o1/v/...,541697.0,541697.0,"{'artist_name': 'Selena Gomez, benny blanco, T...",3,3,Reel 3,This color!!🍵😍 Comment “SHOP” & I’ll send you ...,2025-04-05 19:40:25,33062
3,reel,https://www.instagram.com/p/C9TgmWayIWL/,C9TgmWayIWL,26631,770,28941,2024-07-12 01:51:39,2024.0,2024-07,Coming in hot with some of my favourite single...,...,https://scontent-vie1-1.cdninstagram.com/o1/v/...,5050571.0,5050571.0,"{'artist_name': 'by.vice', 'song_name': 'Origi...",4,4,Reel 4,Coming in hot with some of my favourite single...,2024-07-12 01:51:39,28941
4,reel,https://www.instagram.com/p/DAl5MSDywMJ/,DAl5MSDywMJ,26330,184,26882,2024-10-01 18:51:40,2024.0,2024-10,Why ruin your body with alcohol when you can w...,...,https://scontent-vie1-1.cdninstagram.com/o1/v/...,1337579.0,1337579.0,"{'artist_name': 'Freak Nasty', 'song_name': ""D...",5,5,Reel 5,Why ruin your body with alcohol when you can w...,2024-10-01 18:51:40,26882


## 3. Instagram - Comentarios

In [4]:
ig_com_us = pd.read_csv(f"{DATA_PROC}/instagram/ig_lululemon_comentarios_US.csv")
ig_com_eu = pd.read_csv(f"{DATA_PROC}/instagram/ig_lululemon_comentarios_EUROPE.csv")

ig_com = pd.concat([ig_com_us, ig_com_eu], ignore_index=True)
ig_com["platform"] = "instagram"
ig_com["text_clean"] = ig_com["text"].apply(clean_text)
ig_com["timestamp"] = pd.to_datetime(ig_com["timestamp"], errors="coerce", utc=True).dt.tz_localize(None)

# Eliminar comentarios vacios
n_before = len(ig_com)
ig_com = ig_com[ig_com["text_clean"].str.len() > 2].reset_index(drop=True)
print(f"Instagram comentarios: {n_before} -> {len(ig_com)} (eliminados {n_before - len(ig_com)} vacios)")

# Eliminar duplicados
ig_com = ig_com.drop_duplicates(subset=["comment_id"], keep="first")
print(f"Tras deduplicar: {len(ig_com)}")
ig_com.head()

Instagram comentarios: 1438 -> 1433 (eliminados 5 vacios)
Tras deduplicar: 1433


,post_url,comment_id,text,timestamp,likes,username,lang,region,platform,text_clean
0,https://www.instagram.com/p/DBWeiONpNdr/,17860467027302927,I promise this reel makes more sense if you wa...,2024-12-18 03:15:21,777,hiladailythrift,en,US,instagram,I promise this reel makes more sense if you wa...
1,https://www.instagram.com/p/DBWeiONpNdr/,18043803248155317,Bought all that for a 5 day gym phase,2025-04-15 17:53:24,1,lukeplotts,en,US,instagram,Bought all that for a 5 day gym phase
2,https://www.instagram.com/p/DBWeiONpNdr/,17847238104430594,All polyester clothes stink,2025-03-17 18:17:01,1,fourhareathome,en,US,instagram,All polyester clothes stink
3,https://www.instagram.com/p/DBWeiONpNdr/,18059094641011764,Some white girl died today,2025-03-05 23:52:02,2,thomasscanlon28,en,US,instagram,Some white girl died today
4,https://www.instagram.com/p/DBWeiONpNdr/,17980347599804676,"damn I got a bunch at goodwill for $8, 25 is c...",2025-02-23 15:27:22,2,txri,en,US,instagram,"damn I got a bunch at goodwill for $8, 25 is c..."


## 4. TikTok - Videos

In [5]:
tk_us = pd.read_csv(f"{DATA_PROC}/tiktok/tiktok_lululemon_videos_US.csv")
tk_eu = pd.read_csv(f"{DATA_PROC}/tiktok/tiktok_lululemon_videos_EUROPE.csv")

tk_us["region_search"] = "US"
tk_eu["region_search"] = "Europe"

tk = pd.concat([tk_us, tk_eu], ignore_index=True)

# Limpieza
tk["title_clean"] = tk["title"].apply(clean_text)
tk["uploaded_at"] = pd.to_datetime(tk["uploadedAtFormatted"], errors="coerce")
tk["year"] = tk["uploaded_at"].dt.year
tk["month"] = tk["uploaded_at"].dt.to_period("M").astype(str)

# Engagement rate con views
tk = add_engagement_rate(tk, "likes", "comments", "views")

# Deduplicar por id
n_before = len(tk)
tk = tk.drop_duplicates(subset="id", keep="first")
print(f"TikTok videos: {n_before} -> {len(tk)} (eliminados {n_before - len(tk)} duplicados)")
tk.head()

TikTok videos: 400 -> 295 (eliminados 105 duplicados)


/var/folders/hw/k9dl2bvx033g270sxxkmnwlw0000gn/T/ipykernel_14250/1947926415.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  tk["month"] = tk["uploaded_at"].dt.to_period("M").astype(str)


,inputSource,id,title,views,likes,comments,shares,bookmarks,hashtags,channel,...,author,score,rank,region_search,location_search,title_clean,uploaded_at,year,month,engagement_rate
0,https://www.tiktok.com/tag/lululemonhaul,7200918254892649770,now im ✨obsessed✨ #NextLevelDish #fyp #xyzbca ...,5371145,608149,482,1850,25164,"['nextleveldish', 'fyp', 'xyzbca', 'foryou', '...","{'id': '6713180620846236677', 'name': 'hunts',...",...,huntercamp33,613295,1,US,NaN,now im ✨obsessed✨ #NextLevelDish #fyp #xyzbca ...,2023-02-17 00:40:44+00:00,2023,2023-02,11.331494
1,https://www.tiktok.com/tag/lululemonhaul,7266988919261285678,best day 🫶🏼🫶🏼🫶🏼 thanks mom #haul #hauls #backt...,5543233,579366,2092,7022,33632,"['haul', 'hauls', 'backtoschool', 'backtoschoo...","{'id': '7125849146178929706', 'name': 'lisi:)'...",...,lisi.shops,599686,2,US,NaN,best day 🫶🏼🫶🏼🫶🏼 thanks mom #haul #hauls #backt...,2023-08-14 01:48:39+00:00,2023,2023-08,10.489510
2,https://www.tiktok.com/tag/lululemonfinds,6968899199773576453,#thriftwithtay #lululemonfinds #thriftfinds #fyp,7691473,404703,236,207,2284,"['thriftwithtay', 'lululemonfinds', 'thriftfin...","{'id': '6786819791468135430', 'name': 'Tayla',...",...,tiktokwithtay,405825,3,US,NaN,#thriftwithtay #lululemonfinds #thriftfinds #fyp,2021-06-01 18:48:10+00:00,2021,2021-06,5.264778
3,https://www.tiktok.com/tag/lululemon,7619882392978803981,Come with me to buy my hg the new lululemon ja...,1612464,257312,2692,49366,18963,"['comewithme', 'vlog', 'lululemon', 'lulu', 'f...","{'id': '7391580400458318894', 'name': 'Zay ✯',...",...,8kzayy,364120,4,US,NaN,Come with me to buy my hg the new lululemon ja...,2026-03-22 01:15:43+00:00,2026,2026-03,16.124639
4,https://www.tiktok.com/tag/lululemonoutfit,7119551158422719787,had to hop on this trend 🙈💖 #fakebody #lululem...,2448898,331023,1589,934,11556,"['fakebody', 'lululemon', 'lululemonoutfit', '...","{'id': '6919547932983395334', 'name': 'bellagi...",...,bellagiannop,337658,5,US,NaN,had to hop on this trend 🙈💖 #fakebody #lululem...,2022-07-12 18:14:42+00:00,2022,2022-07,13.582109


## 5. TikTok - Comentarios

In [6]:
tk_com_us = pd.read_csv(f"{DATA_PROC}/tiktok/tiktok_lululemon_comentarios_US.csv")
tk_com_eu = pd.read_csv(f"{DATA_PROC}/tiktok/tiktok_lululemon_comentarios_EUROPE.csv")

tk_com_us["region_search"] = "US"
tk_com_eu["region_search"] = "Europe"

tk_com = pd.concat([tk_com_us, tk_com_eu], ignore_index=True)
tk_com["platform"] = "tiktok"
tk_com["text_clean"] = tk_com["text"].apply(clean_text)
tk_com["timestamp"] = pd.to_datetime(tk_com["createTimeISO"], errors="coerce")

# Renombrar para esquema uniforme
tk_com = tk_com.rename(columns={
    "cid": "comment_id",
    "diggCount": "likes",
    "uniqueId": "username",
})

# Eliminar vacios
n_before = len(tk_com)
tk_com = tk_com[tk_com["text_clean"].str.len() > 2].reset_index(drop=True)
print(f"TikTok comentarios: {n_before} -> {len(tk_com)} (eliminados {n_before - len(tk_com)} vacios)")
tk_com.head()

TikTok comentarios: 1056 -> 1056 (eliminados 0 vacios)


,videoWebUrl,submittedVideoUrl,input,comment_id,createTime,createTimeISO,text,likes,likedByAuthor,pinnedByAuthor,...,username,avatarThumbnail,mentions,detailedMentions,idioma_detectado,region_search,pais_busqueda,platform,text_clean,timestamp
0,https://www.tiktok.com/@rose32xz/video/7552319...,https://www.tiktok.com/@rose32xz/video/7552319...,https://www.tiktok.com/@rose32xz/video/7552319...,7552562618762871566,1758468030,2025-09-21T15:20:30.000Z,"First off, I can't even fit in a size 0 lulule...",14883,False,False,...,nataliesmith852,https://p16-common-sign.tiktokcdn-us.com/tos-u...,[],[],en,US,NaN,tiktok,"First off, I can't even fit in a size 0 lulule...",2025-09-21 15:20:30+00:00
1,https://www.tiktok.com/@rose32xz/video/7552319...,https://www.tiktok.com/@rose32xz/video/7552319...,https://www.tiktok.com/@rose32xz/video/7552319...,7552785898472768311,1758520082,2025-09-22T05:48:02.000Z,She didn’t see the 1 in front of the 0 probably,44030,False,False,...,illyriandoll,https://p16-common-sign.tiktokcdn-us.com/tos-u...,[],[],en,US,NaN,tiktok,She didn’t see the 1 in front of the 0 probably,2025-09-22 05:48:02+00:00
2,https://www.tiktok.com/@rose32xz/video/7552319...,https://www.tiktok.com/@rose32xz/video/7552319...,https://www.tiktok.com/@rose32xz/video/7552319...,7552532950324314910,1758461129,2025-09-21T13:25:29.000Z,Idk how girlies are squeezing into 2 sizes sma...,1563,False,False,...,imliterbeeinsane,https://p16-common-sign.tiktokcdn-us.com/tos-u...,[],[],en,US,NaN,tiktok,Idk how girlies are squeezing into 2 sizes sma...,2025-09-21 13:25:29+00:00
3,https://www.tiktok.com/@rose32xz/video/7552319...,https://www.tiktok.com/@rose32xz/video/7552319...,https://www.tiktok.com/@rose32xz/video/7552319...,7552550536177582861,1758465257,2025-09-21T14:34:17.000Z,Im a size 2 and the size 0 arms are so tight 😭,23179,False,False,...,lbutterfliesl,https://p16-common-sign.tiktokcdn-us.com/tos-u...,[],[],en,US,NaN,tiktok,Im a size 2 and the size 0 arms are so tight 😭,2025-09-21 14:34:17+00:00
4,https://www.tiktok.com/@rose32xz/video/7552319...,https://www.tiktok.com/@rose32xz/video/7552319...,https://www.tiktok.com/@rose32xz/video/7552319...,7552561072814293773,1758467691,2025-09-21T15:14:51.000Z,Girl that size 6 looks fine,20404,True,False,...,a.alimac,https://p16-common-sign.tiktokcdn-us.com/tos-u...,[],[],en,US,NaN,tiktok,Girl that size 6 looks fine,2025-09-21 15:14:51+00:00


## 6. Resumen y guardado de datos limpios

In [7]:
# Guardar
tp.to_csv(f"{DATA_CLEAN}/trustpilot_all.csv", index=False, encoding="utf-8-sig")
ig.to_csv(f"{DATA_CLEAN}/ig_publicaciones.csv", index=False, encoding="utf-8-sig")
ig_com.to_csv(f"{DATA_CLEAN}/ig_comentarios.csv", index=False, encoding="utf-8-sig")
tk.to_csv(f"{DATA_CLEAN}/tiktok_videos.csv", index=False, encoding="utf-8-sig")
tk_com.to_csv(f"{DATA_CLEAN}/tiktok_comentarios.csv", index=False, encoding="utf-8-sig")

import os
os.makedirs("../../outputs/tablas", exist_ok=True)

# Resumen
resumen = pd.DataFrame({
    "Dataset": ["Trustpilot Reviews", "Instagram Publicaciones", "Instagram Comentarios",
                 "TikTok Videos", "TikTok Comentarios"],
    "Filas": [len(tp), len(ig), len(ig_com), len(tk), len(tk_com)],
    "Columnas": [tp.shape[1], ig.shape[1], ig_com.shape[1], tk.shape[1], tk_com.shape[1]],
    "Nulos (%)": [
        round(tp.isnull().mean().mean()*100, 1),
        round(ig.isnull().mean().mean()*100, 1),
        round(ig_com.isnull().mean().mean()*100, 1),
        round(tk.isnull().mean().mean()*100, 1),
        round(tk_com.isnull().mean().mean()*100, 1),
    ],
})

resumen.to_csv("../../outputs/tablas/resumen_datasets.csv", index=False)
print("Datos limpios guardados en datos/clean/")
resumen

Datos limpios guardados en datos/clean/


,Dataset,Filas,Columnas,Nulos (%)
0,Trustpilot Reviews,383,18,0.0
1,Instagram Publicaciones,577,30,10.0
2,Instagram Comentarios,1433,10,0.0
3,TikTok Videos,295,31,15.8
4,TikTok Comentarios,1056,23,8.5
